# Demo 10 — Ascoltare campana, mensola e passa-basso

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 44100
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
b_sh, a_sh = al.eq_shelf(2500, -9, "high")
b_bell, a_bell = al.eq_bell(2500, 1000, -9)
sos_lp = al.design_filter("lowpass", 2500, sr=SR, order=4)
progetti = {
    "Campana −9 dB": signal.tf2sos(b_bell, a_bell),
    "Mensola alte −9 dB": signal.tf2sos(b_sh, a_sh),
    "Passa-basso 2500 Hz": sos_lp,
}
fig = go.Figure()
voce_eq, sr = al.load_audio(AUDIO / "speech1.wav", sr=SR, seconds=4.0)
versioni_eq = [("Originale", voce_eq)]
f_griglia = np.geomspace(30, 20000, 2000)
for nome, sos_eq in progetti.items():
    _, H_eq = signal.sosfreqz(sos_eq, worN=f_griglia, fs=sr)
    fig.add_trace(go.Scatter(x=f_griglia, y=al.to_db(np.abs(H_eq), floor=1e-6), name=nome))
    versioni_eq.append((nome, al.filter_sos(sos_eq, voce_eq)))
fig.update_xaxes(type="log", title_text="Frequenza [Hz]")
fig.update_yaxes(title_text="Guadagno [dB]", range=[-45, 3])
fig.update_layout(height=380, title="Stessa frequenza indicata, tre interventi diversi")
fig.show()
al.compare_audio(versioni_eq, sr)
